# 12F — Point-in-Time Embedding Propagation and Placebos

This notebook uses quarterly LLM-positive firms as seeds and discovers nearby S&P 500 firms in the contemporaneous RS-L-Min embedding space. The neighbour rule, null threshold, cardinality, and placebo design are frozen before any return data are opened.

## 1. Load the frozen design and quarterly inputs

In [1]:
from pathlib import Path
import hashlib,json,numpy as np,pandas as pd
from IPython.display import display
ROOT=Path.cwd().resolve()
if ROOT.name!='sp500_critical_minerals':
    candidate=ROOT/'geopolitical_exposure'/'sp500_critical_minerals'
    if candidate.exists(): ROOT=candidate
PROJECT=ROOT.parents[1]; CONFIG=ROOT/'config'; SOURCE=ROOT/'quarterly_exposure_v1'; OUT=ROOT/'embedding_propagation_v1'; OUT.mkdir(exist_ok=True)
contract_path=CONFIG/'embedding_propagation_v1.json'; contract=json.loads(contract_path.read_text())
quarterly=pd.read_parquet(SOURCE/'quarterly_firm_exposure_panel.parquet')
EMBED_ROOT=PROJECT/'full_replication_80q'/'embeddings'
assert contract['returns_accessed'] is False
print({'firm_quarters':len(quarterly),'positive_seeds':int(quarterly.positive_seed.sum()),'returns_loaded':False})

{'firm_quarters': 2001, 'positive_seeds': 65, 'returns_loaded': False}


## 2. Load and normalize contemporaneous 10-dimensional embeddings

Only the first ten components of each quarter's normalized RS-L-Min artifact are used. Rows are unit-normalized for cosine similarity. Coverage is evaluated separately in every vintage.

In [2]:
embeddings={}; coverage=[]
for vintage in sorted(quarterly.vintage.unique()):
    path=EMBED_ROOT/vintage/'rs_l_min'/'asset_embeddings_normalized_labeled.parquet'; assert path.exists(),path
    frame=pd.read_parquet(path); dims=[f'embedding_{i:03d}' for i in range(1,contract['dimensions']+1)]
    frame=frame[['permco','firm_name']+dims].drop_duplicates('permco').copy(); matrix=frame[dims].to_numpy(float); norms=np.linalg.norm(matrix,axis=1); keep=np.isfinite(matrix).all(1)&(norms>0); frame=frame.loc[keep].reset_index(drop=True); matrix=matrix[keep]/norms[keep,None]
    embeddings[vintage]=(frame,matrix,dims)
    u=quarterly[quarterly.vintage.eq(vintage)]; covered=set(u.permco)&set(frame.permco); seeds=set(u.loc[u.positive_seed,'permco']); coverage.append({'vintage':vintage,'sp500_firms':u.permco.nunique(),'embedded_sp500_firms':len(covered),'embedding_coverage':len(covered)/u.permco.nunique(),'positive_seeds':len(seeds),'covered_positive_seeds':len(seeds&covered),'uncovered_positive_seeds':len(seeds-covered),'embedding_sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
coverage=pd.DataFrame(coverage); display(coverage); coverage.to_parquet(OUT/'embedding_coverage_and_hashes.parquet',index=False)

,vintage,sp500_firms,embedded_sp500_firms,embedding_coverage,positive_seeds,covered_positive_seeds,uncovered_positive_seeds,embedding_sha256
0,2024q1,500,411,0.822000,16,14,2,4cf3d8e18bde8ab93d3257d202b5c557cd63b4f76f8898...
1,2024q2,500,407,0.814000,15,13,2,5f8112502680f404eabadb928f4106f8dda9df34a04fab...
2,2024q3,501,403,0.804391,17,15,2,37bd4a327ba08f6cb61cfa30eabde4246ccb404abe53de...
3,2024q4,500,410,0.820000,17,15,2,8cafa21d9bc7b6769c4d37952cb55d14b75b8418159733...


## 3. Estimate quarter-specific random-pair null thresholds

The null preserves the observed embedding geometry but removes thematic seed identity by sampling distinct firm pairs. This is a proximity benchmark, not a statistical claim that embeddings were generated independently.

In [3]:
rng=np.random.default_rng(contract['random_seed']); null_rows=[]; null_draws={}
for vintage,(frame,matrix,dims) in embeddings.items():
    members=set(quarterly.loc[quarterly.vintage.eq(vintage),'permco']); idx=np.flatnonzero(frame.permco.isin(members).to_numpy()); n=contract['null_pairs']; left=rng.choice(idx,n); right=rng.choice(idx,n); same=left==right
    while same.any(): right[same]=rng.choice(idx,int(same.sum())); same=left==right
    sims=np.einsum('ij,ij->i',matrix[left],matrix[right]); threshold=float(np.quantile(sims,contract['null_quantile'])); null_draws[vintage]=sims
    null_rows.append({'vintage':vintage,'pairs':n,'mean_similarity':sims.mean(),'median_similarity':np.median(sims),'q90':np.quantile(sims,.90),'q95':threshold,'q99':np.quantile(sims,.99)})
null_summary=pd.DataFrame(null_rows); display(null_summary); null_summary.to_parquet(OUT/'random_pair_null_summary.parquet',index=False)

,vintage,pairs,mean_similarity,median_similarity,q90,q95,q99
0,2024q1,100000,0.251234,0.252746,0.711409,0.801689,0.908177
1,2024q2,100000,0.247898,0.250570,0.700859,0.786892,0.895401
2,2024q3,100000,0.234605,0.236065,0.682663,0.775040,0.890632
3,2024q4,100000,0.234300,0.235654,0.679490,0.769176,0.891900


## 4. Nominate and rank embedding-only firms

For each covered positive seed, retain at most five non-seed neighbours above the null threshold. A candidate's ranking score is its strongest similarity multiplied by the corresponding LLM seed score.

In [4]:
nominations=[]; portfolio_rows=[]
for vintage,(frame,matrix,dims) in embeddings.items():
    u=quarterly[quarterly.vintage.eq(vintage)].copy(); members=set(u.permco); seed_scores=u.loc[u.positive_seed].set_index('permco').positive_score.to_dict(); all_seeds=set(seed_scores); common_idx=np.flatnonzero(frame.permco.isin(members).to_numpy()); common_permcos=frame.loc[common_idx,'permco'].to_numpy(); covered_seeds=[s for s in seed_scores if s in set(common_permcos)]; threshold=null_summary.set_index('vintage').loc[vintage,'q95']
    pos={p:i for i,p in enumerate(frame.permco)}
    for seed in covered_seeds:
        sims=np.einsum('ij,j->i',matrix[common_idx],matrix[pos[seed]]); order=np.argsort(-sims)
        accepted=0
        for j in order:
            candidate=int(common_permcos[j]); similarity=float(sims[j])
            if candidate in all_seeds or candidate==seed or similarity<=threshold: continue
            accepted+=1; nominations.append({'vintage':vintage,'seed_permco':seed,'candidate_permco':candidate,'similarity':similarity,'seed_score':seed_scores[seed],'weighted_similarity':similarity*seed_scores[seed],'seed_neighbor_rank':accepted,'null_threshold':threshold})
            if accepted>=contract['neighbors_per_seed']: break
    nom=pd.DataFrame([x for x in nominations if x['vintage']==vintage]); n_target=len(covered_seeds)
    ranked=(nom.groupby('candidate_permco',as_index=False).agg(candidate_score=('weighted_similarity','max'),maximum_similarity=('similarity','max'),mean_similarity=('similarity','mean'),seed_count=('seed_permco','nunique')).sort_values(['candidate_score','seed_count','maximum_similarity','candidate_permco'],ascending=[False,False,False,True]))
    assert len(ranked)>=n_target,f'{vintage}: only {len(ranked)} candidates for target {n_target}'
    chosen=ranked.head(n_target).copy(); chosen['vintage']=vintage; chosen['portfolio']='embedding_only_primary'; chosen['portfolio_weight']=1/n_target
    for seed in covered_seeds: portfolio_rows.append({'vintage':vintage,'portfolio':'llm_only_common_support','permco':seed,'portfolio_weight':1/n_target,'selection_score':seed_scores[seed]})
    for row in chosen.itertuples(): portfolio_rows.append({'vintage':vintage,'portfolio':'embedding_only_primary','permco':row.candidate_permco,'portfolio_weight':row.portfolio_weight,'selection_score':row.candidate_score})
nominations=pd.DataFrame(nominations); portfolios=pd.DataFrame(portfolio_rows)
nominations.to_parquet(OUT/'seed_candidate_nominations.parquet',index=False); portfolios.to_parquet(OUT/'primary_equal_cardinality_portfolios.parquet',index=False)
display(portfolios.groupby(['vintage','portfolio']).agg(firms=('permco','nunique'),weight_sum=('portfolio_weight','sum')).reset_index())

,vintage,portfolio,firms,weight_sum
0,2024q1,embedding_only_primary,14,1.0
1,2024q1,llm_only_common_support,14,1.0
2,2024q2,embedding_only_primary,13,1.0
3,2024q2,llm_only_common_support,13,1.0
4,2024q3,embedding_only_primary,15,1.0
5,2024q3,llm_only_common_support,15,1.0
6,2024q4,embedding_only_primary,15,1.0
7,2024q4,llm_only_common_support,15,1.0


## 5. Build 100 matched placebo propagation portfolios

Each placebo replaces the thematic seeds with an equal number of randomly selected non-seed firms from the same common-support universe, then applies the identical neighbour and cardinality rules.

In [5]:
placebos=[]; rng=np.random.default_rng(contract['random_seed']+1)
for vintage,(frame,matrix,dims) in embeddings.items():
    u=quarterly[quarterly.vintage.eq(vintage)]; members=set(u.permco); real_seeds=set(u.loc[u.positive_seed,'permco']); idx=np.flatnonzero(frame.permco.isin(members).to_numpy()); permcos=frame.loc[idx,'permco'].to_numpy(); n=int(coverage.set_index('vintage').loc[vintage,'covered_positive_seeds']); threshold=null_summary.set_index('vintage').loc[vintage,'q95']; pool=np.array(sorted(set(permcos)-real_seeds))
    local={p:i for i,p in enumerate(frame.permco)}
    for draw in range(contract['placebo_portfolios']):
        pseudo=set(rng.choice(pool,n,replace=False).tolist()); scores={}
        for seed in pseudo:
            sims=np.einsum('ij,j->i',matrix[idx],matrix[local[seed]]); eligible=[(int(permcos[j]),float(sims[j])) for j in np.argsort(-sims) if int(permcos[j]) not in pseudo and float(sims[j])>threshold][:contract['neighbors_per_seed']]
            for candidate,sim in eligible: scores[candidate]=max(scores.get(candidate,-np.inf),sim)
        chosen=sorted(scores.items(),key=lambda x:(-x[1],x[0]))[:n]; assert len(chosen)==n
        for candidate,score in chosen: placebos.append({'vintage':vintage,'placebo_draw':draw,'permco':candidate,'portfolio_weight':1/n,'selection_score':score})
placebos=pd.DataFrame(placebos); placebos.to_parquet(OUT/'matched_placebo_portfolios_100.parquet',index=False)
display(placebos.groupby('vintage').agg(draws=('placebo_draw','nunique'),rows=('permco','size'),firms_per_draw=('permco',lambda x:len(x)/contract['placebo_portfolios'])).reset_index())

,vintage,draws,rows,firms_per_draw
0,2024q1,100,1400,14.0
1,2024q2,100,1300,13.0
2,2024q3,100,1500,15.0
3,2024q4,100,1500,15.0


## 6. Freeze decision gates and handoff

These artifacts define selection before returns. The next stage may load outcomes only if coverage, cardinality, threshold, overlap, and placebo gates all pass.

In [6]:
counts=portfolios.groupby(['vintage','portfolio']).permco.nunique().unstack(); overlap=portfolios.groupby('vintage').apply(lambda g:len(set(g.loc[g.portfolio.eq('llm_only_common_support'),'permco'])&set(g.loc[g.portfolio.eq('embedding_only_primary'),'permco'])),include_groups=False)
gate=pd.DataFrame([{'gate':'four_vintages','value':coverage.vintage.nunique(),'threshold':4,'passed':coverage.vintage.nunique()==4},{'gate':'minimum_embedding_coverage','value':coverage.embedding_coverage.min(),'threshold':.75,'passed':coverage.embedding_coverage.min()>=.75},{'gate':'equal_primary_cardinality','value':int((counts.iloc[:,0]==counts.iloc[:,1]).sum()),'threshold':4,'passed':(counts.iloc[:,0]==counts.iloc[:,1]).all()},{'gate':'zero_seed_candidate_overlap','value':int(overlap.sum()),'threshold':0,'passed':overlap.sum()==0},{'gate':'one_hundred_placebos_each','value':int(placebos.groupby('vintage').placebo_draw.nunique().min()),'threshold':100,'passed':placebos.groupby('vintage').placebo_draw.nunique().min()==100},{'gate':'returns_unopened','value':0,'threshold':0,'passed':True}])
display(gate); gate.to_parquet(OUT/'embedding_propagation_gates.parquet',index=False); assert gate.passed.all()
manifest={'stage':'embedding_propagation_v1','contract_sha256':hashlib.sha256(contract_path.read_bytes()).hexdigest(),'vintages':4,'nominations':len(nominations),'primary_portfolio_rows':len(portfolios),'placebo_rows':len(placebos),'returns_accessed':False}
(OUT/'embedding_propagation_manifest.json').write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n'); print(json.dumps(manifest,indent=2))

,gate,value,threshold,passed
0,four_vintages,4.000000,4.00,True
1,minimum_embedding_coverage,0.804391,0.75,True
2,equal_primary_cardinality,4.000000,4.00,True
3,zero_seed_candidate_overlap,0.000000,0.00,True
4,one_hundred_placebos_each,100.000000,100.00,True
5,returns_unopened,0.000000,0.00,True


{
  "stage": "embedding_propagation_v1",
  "contract_sha256": "27c319648c2528b4e7a167e5e11c647695e21f1decc00c1a2757bd2385db2878",
  "vintages": 4,
  "nominations": 262,
  "primary_portfolio_rows": 114,
  "placebo_rows": 5700,
  "returns_accessed": false
}
